In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv('../data/raw/employee_engagement_survey_data.csv') 

In [4]:
df.head()

,Employee ID,Survey Date,Engagement Score,Satisfaction Score,Work-Life Balance Score
0,1001,10-10-2022,2,5,5
1,1002,03-08-2023,4,5,3
2,1003,03-01-2023,2,5,2
3,1004,30-07-2023,3,5,3
4,1005,19-06-2023,2,4,5


In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
df.shape
df.columns.tolist()
df.info()
df.head()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 5 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Employee ID              3000 non-null   int64 
 1   Survey Date              3000 non-null   object
 2   Engagement Score         3000 non-null   int64 
 3   Satisfaction Score       3000 non-null   int64 
 4   Work-Life Balance Score  3000 non-null   int64 
dtypes: int64(4), object(1)
memory usage: 117.3+ KB


,Employee ID,Engagement Score,Satisfaction Score,Work-Life Balance Score
count,3000.000000,3000.000000,3000.000000,3000.000000
mean,2500.500000,2.939667,3.022000,2.989000
std,866.169729,1.433426,1.408845,1.409329
min,1001.000000,1.000000,1.000000,1.000000
25%,1750.750000,2.000000,2.000000,2.000000
50%,2500.500000,3.000000,3.000000,3.000000
75%,3250.250000,4.000000,4.000000,4.000000
max,4000.000000,5.000000,5.000000,5.000000


In [7]:
df.columns = [
    'employee_id',
    'survey_date',
    'engagement_score',
    'satisfaction_score',
    'work_life_balance_score'
]

In [8]:
df.columns.tolist()

['employee_id',
 'survey_date',
 'engagement_score',
 'satisfaction_score',
 'work_life_balance_score']

In [9]:
df['employee_id'].is_unique

True

### PostgreSQL Import Error – Column Order Mismatch

**Issue:** During CSV import, PostgreSQL returned an `invalid input syntax for type integer` error because a date value (`10-10-2022`) was being inserted into the `employee_id` column.

**Root Cause:** The CSV column order did not match the PostgreSQL table structure. PostgreSQL maps CSV values by position rather than matching column names.

**Resolution:**
- Inspected the CSV column names and data.
- Standardized column names to lowercase snake_case.
- Reordered columns to match the PostgreSQL table schema.
- Validated data types and exported the corrected CSV.

**Learning:** Always verify CSV column order and data types before importing data into PostgreSQL.

In [10]:
print("Actual CSV columns:")
print(df.columns.tolist())

print("\nFirst 5 records:")
display(df.head())

Actual CSV columns:
['employee_id', 'survey_date', 'engagement_score', 'satisfaction_score', 'work_life_balance_score']

First 5 records:


,employee_id,survey_date,engagement_score,satisfaction_score,work_life_balance_score
0,1001,10-10-2022,2,5,5
1,1002,03-08-2023,4,5,3
2,1003,03-01-2023,2,5,2
3,1004,30-07-2023,3,5,3
4,1005,19-06-2023,2,4,5


In [11]:
# Standardize column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

# Define required column order
expected_columns = [
    "employee_id",
    "survey_date",
    "engagement_score",
    "satisfaction_score",
    "work_life_balance_score"
]

# Reorder columns
df = df[expected_columns]

# Verify
print(df.columns.tolist())
display(df.head())

['employee_id', 'survey_date', 'engagement_score', 'satisfaction_score', 'work_life_balance_score']


,employee_id,survey_date,engagement_score,satisfaction_score,work_life_balance_score
0,1001,10-10-2022,2,5,5
1,1002,03-08-2023,4,5,3
2,1003,03-01-2023,2,5,2
3,1004,30-07-2023,3,5,3
4,1005,19-06-2023,2,4,5


In [13]:
# Validate employee IDs
df["employee_id"] = pd.to_numeric(
    df["employee_id"],
    errors="raise"
).astype("Int64")

# Export
df.to_csv(
    "../data/cleaned/employee_engagement_cleaned.csv",
    index=False,
    encoding="utf-8"
)